### Setup toy data

In [122]:
import numpy as np
from numpy.typing import NDArray
from scipy.stats import norm
from tqdm import tqdm
import matplotlib.pyplot as plt

FloatArray = NDArray[np.float64]


def generate_toy_data(
    num_samples: int,
    feature_dim: int = 256,
    num_ground_truth_features: int = 512,
    num_active_features: float = 5.0,
    decay_rate: float = 0.99,
    random_seed: int | None = None,
) -> tuple[FloatArray, FloatArray, FloatArray, FloatArray]:
    """
    Generate toy data for sparse feature-learning experiments.

    Generation steps:
    1. Sample `num_ground_truth_features` random directions in R^{feature_dim}
       and normalize each one to unit norm.
    2. Build a random positive-semidefinite covariance matrix across latent
       features.
    3. Sample correlated Gaussian latents for each synthetic sample.
    4. Map those Gaussian latents through the standard normal CDF to get
       correlated values in (0, 1).
    5. Apply index-based decay so lower-index features are generally more
       common.
    6. Rescale probabilities so the expected number of active features is
       `num_active_features`.
    7. Draw Bernoulli support masks and assign Uniform(0, 1) amplitudes to
       active features.
    8. Form each observed sample as a linear combination of normalized ground-
       truth feature directions.

    Returns
    -------
    dataset : NDArray[np.float64], shape (num_samples, feature_dim)
    ground_truth_features : NDArray[np.float64], shape (feature_dim, num_ground_truth_features)
    sparse_coefficients : NDArray[np.float64], shape (num_samples, num_ground_truth_features)
    final_probabilities : NDArray[np.float64], shape (num_samples, num_ground_truth_features)
    """

    # Use assert statements for basic input validation, with informative error messages.
    assert num_samples > 0, f"num_samples must be positive, got {num_samples}."
    assert feature_dim > 0, f"feature_dim must be positive, got {feature_dim}."
    assert (
        num_ground_truth_features > 0
    ), f"num_ground_truth_features must be positive, got {num_ground_truth_features}."
    assert (
        num_active_features >= 0
    ), f"num_active_features must be nonnegative, got {num_active_features}."
    assert decay_rate > 0, f"decay_rate must be positive, got {decay_rate}."

    rng = np.random.default_rng(random_seed)

    # ------------------------------------------------------------------
    # 1) Sample ground-truth feature directions, then normalize columns.
    #    Shape: (feature_dim, num_ground_truth_features)
    # ------------------------------------------------------------------
    ground_truth_features: FloatArray = rng.standard_normal(
        size=(feature_dim, num_ground_truth_features)
    )
    column_norms: FloatArray = np.linalg.norm(ground_truth_features, axis=0)
    column_norms = np.where(column_norms == 0.0, 1.0, column_norms)
    ground_truth_features = ground_truth_features / column_norms

    # ------------------------------------------------------------------
    # 2) Build a random covariance matrix for correlated latent features.
    # ------------------------------------------------------------------
    covariance_seed: FloatArray = rng.standard_normal(
        size=(num_ground_truth_features, num_ground_truth_features)
    )
    covariance: FloatArray = covariance_seed @ covariance_seed.T

    # ------------------------------------------------------------------
    # 3) Draw correlated Gaussian latents for all samples at once.
    # ------------------------------------------------------------------
    mean: FloatArray = np.zeros(num_ground_truth_features, dtype=np.float64)
    gaussian_samples: FloatArray = rng.multivariate_normal(
        mean,
        covariance,
        size=num_samples,
    )

    # ------------------------------------------------------------------
    # 4) Map Gaussian samples through the standard normal CDF.
    # ------------------------------------------------------------------
    correlated_feature_probs: FloatArray = norm.cdf(gaussian_samples)

    # ------------------------------------------------------------------
    # 5) Allocate outputs and precompute feature indices for decay.
    # ------------------------------------------------------------------
    sparse_coefficients: FloatArray = np.zeros(
        (num_samples, num_ground_truth_features), dtype=np.float64
    )
    dataset: FloatArray = np.zeros((num_samples, feature_dim), dtype=np.float64)
    
    feature_indices: FloatArray = np.arange(num_ground_truth_features, dtype=np.float64)

    # ------------------------------------------------------------------
    # 6) Per sample, apply decay and rescale probabilities to match the
    #    desired expected active-feature count.
    # 7) Sample Bernoulli support and Uniform(0,1) amplitudes.
    # 8) Form observed vector by linear combination of latent features.
    # ------------------------------------------------------------------
    for i in tqdm(range(num_samples)):
        decayed_feature_probs: FloatArray = np.power(
            correlated_feature_probs[i],
            feature_indices * decay_rate,
        )

        mean_prob = float(np.mean(decayed_feature_probs))
        if mean_prob <= 0.0:
            raise ValueError(
                "Mean decayed probability became non-positive, which should "
                "not happen with valid inputs."
            )

        rescaled_probs: FloatArray = decayed_feature_probs / mean_prob
        rescaled_probs = (
            num_active_features * rescaled_probs / num_ground_truth_features
        )

        binary_vector: FloatArray = rng.binomial(1, rescaled_probs).astype(np.float64)
        activations: FloatArray = binary_vector * rng.uniform(
            0.0,
            1.0,
            num_ground_truth_features,
        )

        sparse_coefficients[i] = activations
        dataset[i] = ground_truth_features @ activations

    return dataset, ground_truth_features, sparse_coefficients

In [123]:
# Setup deterministic seed
SEED_NUM = 42

def setup_seed(seed: int) -> None:
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

setup_seed(SEED_NUM)

In [124]:
# Generate the toy dataset with N samples
num_samples = 100000
feature_dim = 256
num_ground_truth_features = 512
num_active_features = 5
dataset, ground_truth_features, sparse_coefficients = generate_toy_data(
    num_samples=num_samples,
    feature_dim=feature_dim,
    num_ground_truth_features=num_ground_truth_features,
    num_active_features=num_active_features,
    decay_rate=0.99,
    random_seed=SEED_NUM,)

100%|██████████| 100000/100000 [00:03<00:00, 27283.48it/s]


In [125]:
# Dataset is the activations per sample
dataset.shape

(100000, 256)

#### Setup SNMF model

In [126]:
import torch as t
import torch.nn as nn
from dataclasses import dataclass
from typing import List, Optional

def set_seed(seed: int) -> None:
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def positive_part(X: t.Tensor) -> t.Tensor:
    return 0.5 * (X.abs() + X)

def negative_part(X: t.Tensor) -> t.Tensor:
    return 0.5 * (X.abs() - X)

@t.no_grad()
def wta_sparsify_inplace(Z: t.Tensor, pct_keep: float) -> None:
    """
    Winner-Take-All sparsification on columns of Z.
    Keeps only top `pct_keep` fraction by absolute value in each column.
    Mutates Z in-place.
    """
    if pct_keep <= 0 or pct_keep > 1:
        raise ValueError("pct_keep must be in (0, 1].")

    d, K = Z.shape
    k = max(1, int(t.ceil(t.tensor(pct_keep * d)).item()))

    scores = Z.abs()
    topk_vals, _ = t.topk(scores, k, dim=0, largest=True, sorted=True)
    thresh = topk_vals[-1, :]  # (K,)
    mask = scores >= thresh.unsqueeze(0)  # (d, K)
    Z.mul_(mask)

@t.no_grad()
def fix_scale_inplace(Z: t.Tensor, Y: t.Tensor, eps: float = 1e-8) -> None:
    """
    Normalize columns of Y to unit L2 norm, and compensate by scaling columns of Z,
    so that Z @ Y.T stays unchanged.
    """
    col_norms = Y.norm(dim=0, keepdim=True).clamp_min(eps)  # (1, K)
    Y.div_(col_norms)
    Z.mul_(col_norms.squeeze(0))  # scale each column of Z

In [127]:
from dataclasses import dataclass
from typing import List, Optional
import torch as t
import torch.nn as nn

@dataclass
class SemiNMFResult:
    Z: t.Tensor                  # (d_hidden, K)
    Y: t.Tensor                  # (N, K)
    loss_history: List[float]
    best_iter: int
    best_loss: float


class SemiNMF(nn.Module):
    """
    Semi-NMF factorization: A ≈ Z @ Y.T, with Y >= 0 and Z unconstrained.

    Designed for interpretability experiments:
      - optional WTA sparsification on Z
      - scale stabilization
      - early stopping
      - device-safe inverse/solve
    """

    def __init__(
        self,
        K: int,
        device: t.device,
        dtype: t.dtype = t.float32,
        sparsity_keep: float = 1.0,   # 1.0 means no sparsification
    ):
        super().__init__()
        if K <= 0:
            raise ValueError("K must be positive.")
        self.K = K
        self.device = device
        self.dtype = dtype
        self.sparsity_keep = sparsity_keep

        self.Z_: Optional[t.Tensor] = None
        self.Y_: Optional[t.Tensor] = None

    @t.no_grad()
    def _full_loss_samples_by_features(
        self,
        X: t.Tensor,   # (N, d_hidden)
        Y: t.Tensor,   # (N, K)
        Z: t.Tensor,   # (d_hidden, K)
        batch_size: int,
    ) -> float:
        total = 0.0
        N = X.shape[0]
        for start in range(0, N, batch_size):
            end = min(N, start + batch_size)
            Xb = X[start:end]
            Yb = Y[start:end]
            diff = Xb - (Yb @ Z.T)
            total += (diff * diff).sum().item()
        return total

    @t.no_grad()
    def fit(
        self,
        A: t.Tensor,                 # (d_hidden, N)
        max_iter: int = 1000,
        tol: float = 1e-6,
        reg: float = 1e-6,
        closed_form_eqn_reg: float = 1e-6,
        sparsity_reg: float = 1e-4,
        patience: int = 50,
        verbose_every: int = 50,
        use_hoyer_normalization: bool = False,
        seed: int = 42,
    ) -> SemiNMFResult:
        """
        Fit Semi-NMF to activation matrix A.

        Args:
            A: (d_hidden, N)
            max_iter: max alternating updates.
            tol: minimum improvement to reset patience.
            reg: ridge regularization for stable inverse in Z update.
            patience: early stop after this many non-improving iterations.
            verbose_every: log period.
            seed: deterministic init.

        Returns:
            SemiNMFResult
        """
        set_seed(seed)

        A = A.to(device=self.device, dtype=self.dtype)
        d_hidden, N = A.shape
        K = self.K

        # Initialize Y >= 0 and Z unconstrained
        Y = t.rand((N, K), device=self.device, dtype=self.dtype).clamp_min(reg)
        Z = t.randn((d_hidden, K), device=self.device, dtype=self.dtype)

        best_loss = float("inf")
        best_it = -1
        best_Z = None
        best_Y = None
        no_improve = 0
        history: List[float] = []

        for it in range(max_iter):
            # ---- (1) Update Z in closed form: Z = A Y (Y^T Y + reg I)^-1
            YtY = Y.T @ Y  # (K, K)
            Z = t.linalg.solve(
                YtY + closed_form_eqn_reg * t.eye(K, device=self.device, dtype=self.dtype),
                (A @ Y).T,
            ).T

            # Optional: sparsify Z for interpretability
            if self.sparsity_keep < 1.0:
                wta_sparsify_inplace(Z, pct_keep=self.sparsity_keep)

            # Scale stabilization (keeps product ZY^T ~ invariant under rescaling)
            if use_hoyer_normalization:
                fix_scale_inplace(Z, Y)

            # ---- (2) Multiplicative update for Y (keeps Y >= 0)
            P = A.T @ Z             # (N, K)
            Q = Z.T @ Z             # (K, K)

            P_plus, P_minus = positive_part(P), negative_part(P)
            Q_plus, Q_minus = positive_part(Q), negative_part(Q)

            numer = P_plus + (Y @ Q_minus)
            denom = P_minus + (Y @ Q_plus)

            # stabilize division
            Y = Y * t.sqrt(numer / (denom + 1e-8))
            Y = Y.clamp_min(1e-8)

            # ---- Loss
            A_hat = Z @ Y.T
            reconstruction_loss = t.norm(A - A_hat, p="fro").pow(2).item()
            sparsity_loss = sparsity_reg * Y.abs().mean().item()
            loss = reconstruction_loss + sparsity_loss

            history.append(loss)

            # ---- Early stopping
            if loss < best_loss - tol:
                best_loss = loss
                best_it = it
                best_Z = Z.detach().clone()
                best_Y = Y.detach().clone()
                no_improve = 0
            else:
                no_improve += 1

            if (verbose_every > 0) and (it % verbose_every == 0 or no_improve == 1):
                print(f"[SemiNMF] iter={it:4d}  loss={loss:.6e}  best={best_loss:.6e}  no_improve={no_improve}")

            if no_improve >= patience:
                print(f"[SemiNMF] early stop at iter={it} (best_iter={best_it}, best_loss={best_loss:.6e})")
                break

        # Restore best
        assert best_Z is not None and best_Y is not None
        self.Z_ = best_Z
        self.Y_ = best_Y

        return SemiNMFResult(
            Z=self.Z_,
            Y=self.Y_,
            loss_history=history,
            best_iter=best_it,
            best_loss=best_loss,
        )

    @t.no_grad()
    def fit_batched(
        self,
        A: t.Tensor,
        *,
        batch_size: int = 2048,
        max_epochs: int = 300,
        y_inner_steps: int = 1,
        tol: float = 1e-6,
        reg: float = 1e-6,
        patience: int = 30,
        verbose_every: int = 20,
        seed: int = 42,
        use_hoyer_normalization: bool = False,
        input_layout: str = "samples_by_features",  # or "features_by_samples"
    ) -> SemiNMFResult:
        """
        Batched Semi-NMF using epoch-wise alternating updates.

        Layouts:
          - samples_by_features: A is (N, d_hidden)
          - features_by_samples: A is (d_hidden, N)

        The Z update is exact per epoch by accumulating sufficient statistics:
          XtY = X^T Y, YtY = Y^T Y, then Z = XtY (YtY + reg I)^-1.
        """
        if batch_size <= 0:
            raise ValueError("batch_size must be positive.")
        if y_inner_steps <= 0:
            raise ValueError("y_inner_steps must be positive.")

        set_seed(seed)

        A = A.to(device=self.device, dtype=self.dtype)

        if input_layout == "samples_by_features":
            X = A
        elif input_layout == "features_by_samples":
            X = A.T
        else:
            raise ValueError("input_layout must be 'samples_by_features' or 'features_by_samples'.")

        if X.ndim != 2:
            raise ValueError(f"Expected 2D input, got shape {tuple(X.shape)}.")

        N, d_hidden = X.shape
        K = self.K

        Y = t.rand((N, K), device=self.device, dtype=self.dtype).clamp_min(reg)
        Z = t.randn((d_hidden, K), device=self.device, dtype=self.dtype)

        best_loss = float("inf")
        best_it = -1
        best_Z = None
        best_Y = None
        no_improve = 0
        history: List[float] = []

        I = t.eye(K, device=self.device, dtype=self.dtype)

        for epoch in range(max_epochs):
            perm = t.randperm(N, device=self.device)

            XtY = t.zeros((d_hidden, K), device=self.device, dtype=self.dtype)
            YtY = t.zeros((K, K), device=self.device, dtype=self.dtype)

            Q = Z.T @ Z
            Q_plus, Q_minus = positive_part(Q), negative_part(Q)

            for start in range(0, N, batch_size):
                end = min(N, start + batch_size)
                idx = perm[start:end]

                Xb = X[idx]              # (b, d_hidden)
                Yb = Y[idx]              # (b, K)

                P = Xb @ Z               # (b, K)
                P_plus, P_minus = positive_part(P), negative_part(P)

                for _ in range(y_inner_steps):
                    numer = P_plus + (Yb @ Q_minus)
                    denom = P_minus + (Yb @ Q_plus)
                    Yb = Yb * t.sqrt(numer / (denom + 1e-8))
                    Yb = Yb.clamp_min(1e-8)

                Y[idx] = Yb

                XtY += Xb.T @ Yb
                YtY += Yb.T @ Yb

            Z = t.linalg.solve(YtY + reg * I, XtY.T).T

            if self.sparsity_keep < 1.0:
                wta_sparsify_inplace(Z, pct_keep=self.sparsity_keep)

            fix_scale_inplace(Z, Y)

            loss = self._full_loss_samples_by_features(X, Y, Z, batch_size=batch_size)
            history.append(loss)

            if loss < best_loss - tol:
                best_loss = loss
                best_it = epoch
                best_Z = Z.detach().clone()
                best_Y = Y.detach().clone()
                no_improve = 0
            else:
                no_improve += 1

            if (verbose_every > 0) and (epoch % verbose_every == 0 or no_improve == 1):
                print(
                    f"[SemiNMF-batch] epoch={epoch:4d}  loss={loss:.6e}  "
                    f"best={best_loss:.6e}  no_improve={no_improve}"
                )

            if no_improve >= patience:
                print(
                    f"[SemiNMF-batch] early stop at epoch={epoch} "
                    f"(best_epoch={best_it}, best_loss={best_loss:.6e})"
                )
                break

        assert best_Z is not None and best_Y is not None
        self.Z_ = best_Z
        self.Y_ = best_Y

        return SemiNMFResult(
            Z=self.Z_,
            Y=self.Y_,
            loss_history=history,
            best_iter=best_it,
            best_loss=best_loss,
        )


In [128]:
# Setup device, let's use mps
device = t.device("mps" if t.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [129]:
A = t.tensor(dataset, dtype=t.float32, device=device)  # (num_samples, d_hidden)
K  = int(2 * num_ground_truth_features)  # Let's start with K = number of ground-truth features
model = SemiNMF(K=K, device=device, dtype=t.float32)
# res = model.fit_batched(
#     A,
#     batch_size=2048,
#     max_epochs=300,
#     y_inner_steps=1,
#     input_layout="samples_by_features",
# )

# Let's try non-batched first to see if it runs at all
res = model.fit(A.T,
                max_iter=500,
                tol=1e-6,
                closed_form_eqn_reg=1e-6,
                sparsity_reg=1e-4,
                patience=10)

[SemiNMF] iter=   0  loss=1.416530e+05  best=1.416530e+05  no_improve=0
[SemiNMF] iter=  50  loss=7.433092e+03  best=7.433092e+03  no_improve=0
[SemiNMF] iter= 100  loss=2.798679e+03  best=2.798679e+03  no_improve=0
[SemiNMF] iter= 150  loss=1.297076e+03  best=1.297076e+03  no_improve=0
[SemiNMF] iter= 200  loss=6.384026e+02  best=6.384026e+02  no_improve=0
[SemiNMF] iter= 250  loss=3.181114e+02  best=3.181114e+02  no_improve=0
[SemiNMF] iter= 300  loss=1.569549e+02  best=1.569549e+02  no_improve=0
[SemiNMF] iter= 350  loss=7.605521e+01  best=7.605521e+01  no_improve=0
[SemiNMF] iter= 400  loss=3.627334e+01  best=3.627334e+01  no_improve=0
[SemiNMF] iter= 450  loss=1.715893e+01  best=1.715893e+01  no_improve=0


In [130]:
snmf_learned_features = res.Z.cpu().numpy()

In [131]:
def get_mean_max_cosine_similarity(learned_features, ground_truth_features):
    """
    Calculates mean max cosine similarity.
    """
    cos_sims = np.dot(ground_truth_features.T, learned_features) / (
        np.linalg.norm(ground_truth_features, axis=0)[:, None] * np.linalg.norm(learned_features, axis=0)
    )
    largest_cosine_similarity = np.max(cos_sims, axis=1)
    mean_cos_sim = np.mean(largest_cosine_similarity)
    return mean_cos_sim


mean_cos_sim = get_mean_max_cosine_similarity(snmf_learned_features, ground_truth_features)
print(f"Mean max cosine similarity: {mean_cos_sim:.6f}")

Mean max cosine similarity: 0.246927


## Results

- Using fit_batched with sparsity 0.1
  - 1G = 0.213
  - 1.2G = 0.216

- Using fit and sparsity 0.1
  - 1G = 0.212
  - 1.2G = 0.217
- Using fit method and sparsity = 1
  - 1.2G = 0.241
  - 2G = 0.247
- Even with sparsity loss, so substantial improvement in similarity (1000 iterations/epochs)
  - 2G = 0.247
  - 2G = 0.247    (without hoyer normalization and 500 epochs)